In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df = pd.read_csv('../data/processed.csv')

model = lgb.Booster(model_file='../results/risk_classifier_lgb.txt')

target = '흡연여부'
metabolic_cols = ['기준_허리둘레', '기준_혈압', '기준_혈당', '기준_트리글리세라이드', '기준_HDL',
                   '대사증후군_점수', '대사증후군_위험군']
drop_cols = ['id', target] + metabolic_cols
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
X['성별'] = X['성별'].map({'남': 1, '여': 0})
X['시력_좌'] = X['시력_좌'].fillna(X['시력_좌'].mean())
X['시력_우'] = X['시력_우'].fillna(X['시력_우'].mean())

y_pred_proba = model.predict(X)

In [ ]:
# 검진 항목별 정상범위 기준표 

reference_ranges = {
    '수축기혈압': (90, 120, '수축기혈압'),
    '이완기혈압': (60, 80, '이완기혈압'),
    '공복혈당': (70, 100, '공복혈당'),
    '총콜레스테롤': (0, 200, '총콜레스테롤'),
    '트리글리세라이드': (0, 150, '중성지방'),
    'HDL콜레스테롤': (40, 999, 'HDL콜레스테롤'),
    'LDL콜레스테롤': (0, 130, 'LDL콜레스테롤'),
    '혈색소': (12, 17, '혈색소'),
    'AST': (0, 40, 'AST(간기능)'),
    'ALT': (0, 40, 'ALT(간기능)'),
    '감마지티피': (0, 50, '감마지티피(간기능)'),
    '혈청크레아티닌': (0.6, 1.2, '혈청크레아티닌(신장기능)'),
    'BMI': (18.5, 25, 'BMI'),
}

In [ ]:
# 항목별 상태 판정 함수 =====
def evaluate_value(col, value):
    low, high, label = reference_ranges[col]

    if col == 'HDL콜레스테롤':
        if value < low:
            return label, value, '낮음'
        return label, value, '정상'

    if value < low:
        return label, value, '낮음'
    elif value > high:
        return label, value, '높음'
    else:
        return label, value, '정상'

In [ ]:
# SHAP 기반 주요 변수 추출 =====
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

if isinstance(shap_values, list):
    shap_values_plot = shap_values[1]
else:
    shap_values_plot = shap_values


def get_top_features(idx, n=5):
    shap_row = pd.Series(shap_values_plot[idx], index=feature_cols).sort_values(key=abs, ascending=False)
    return shap_row.head(n)

In [ ]:
# 소견문 템플릿 생성
def generate_report(idx):
    sample = X.iloc[idx]
    proba = y_pred_proba[idx]
    top_features = get_top_features(idx, n=5)

    lines = []

    # 1) 종합 판정
    if proba >= 0.5:
        lines.append(f"검진 결과 패턴상 흡연 가능성이 높은 것으로 분석됩니다 (예측 확률 {proba:.0%}).")
    else:
        lines.append(f"검진 결과 패턴상 비흡연자에 가까운 것으로 분석됩니다 (예측 확률 {proba:.0%}).")

    # 2) 정상범위 기준 항목별 소견
    abnormal_items = []
    for col in reference_ranges:
        if col in sample.index:
            label, value, status = evaluate_value(col, sample[col])
            if status != '정상':
                abnormal_items.append((label, value, status))

    if abnormal_items:
        lines.append("\n[주의가 필요한 항목]")
        for label, value, status in abnormal_items:
            lines.append(f" - {label}: {value:.1f} ({status})")
    else:
        lines.append("\n주요 검진 항목은 모두 정상범위 내에 있습니다.")

    # 3) 모델 판단에 영향을 준 주요 변수 (SHAP)
    lines.append("\n[분석에 영향을 준 주요 요인]")
    for feat, val in top_features.items():
        direction = "흡연 가능성을 높이는 방향" if val > 0 else "흡연 가능성을 낮추는 방향"
        feat_value = sample[feat]

        if feat == '성별':
            feat_value_display = '남성' if feat_value == 1 else '여성'
        elif feat in ['치석', '치아우식증', '구강검사여부']:
            feat_value_display = '있음' if feat_value == 1 else '없음'
        else:
            feat_value_display = f"{feat_value:.1f}"

        lines.append(f" - {feat} ({feat_value_display}) → {direction}으로 작용")

    # 4) 권고사항
    lines.append("\n[권고사항]")
    if abnormal_items:
        lines.append(" - 주의가 필요한 항목에 대해 생활습관 개선 또는 추가 진료를 권장합니다.")
    if proba >= 0.5:
        lines.append(" - 흡연 관련 건강 위험을 줄이기 위해 금연 및 정기적인 간기능/혈압 검사를 권장합니다.")

    return "\n".join(lines)


검진 결과 패턴상 비흡연자에 가까운 것으로 분석됩니다 (예측 확률 10%).

[주의가 필요한 항목]
 - 총콜레스테롤: 215.0 (높음)

[분석에 영향을 준 주요 요인]
 - 성별 (여성) → 흡연 가능성을 낮추는 방향으로 작용
 - 연령 (40.0) → 흡연 가능성을 높이는 방향으로 작용
 - 감마지티피 (27.0) → 흡연 가능성을 높이는 방향으로 작용
 - 혈색소 (12.9) → 흡연 가능성을 낮추는 방향으로 작용
 - 치석 (있음) → 흡연 가능성을 높이는 방향으로 작용

[권고사항]
 - 주의가 필요한 항목에 대해 생활습관 개선 또는 추가 진료를 권장합니다.
저장 완료: results/sample_reports.csv

===== ID: 0 =====
검진 결과 패턴상 비흡연자에 가까운 것으로 분석됩니다 (예측 확률 10%).

[주의가 필요한 항목]
 - 총콜레스테롤: 215.0 (높음)

[분석에 영향을 준 주요 요인]
 - 성별 (여성) → 흡연 가능성을 낮추는 방향으로 작용
 - 연령 (40.0) → 흡연 가능성을 높이는 방향으로 작용
 - 감마지티피 (27.0) → 흡연 가능성을 높이는 방향으로 작용
 - 혈색소 (12.9) → 흡연 가능성을 낮추는 방향으로 작용
 - 치석 (있음) → 흡연 가능성을 높이는 방향으로 작용

[권고사항]
 - 주의가 필요한 항목에 대해 생활습관 개선 또는 추가 진료를 권장합니다.

===== ID: 1 =====
검진 결과 패턴상 비흡연자에 가까운 것으로 분석됩니다 (예측 확률 5%).

[주의가 필요한 항목]
 - 공복혈당: 130.0 (높음)

[분석에 영향을 준 주요 요인]
 - 성별 (여성) → 흡연 가능성을 낮추는 방향으로 작용
 - 혈색소 (12.7) → 흡연 가능성을 낮추는 방향으로 작용
 - 연령 (40.0) → 흡연 가능성을 높이는 방향으로 작용
 - 감마지티피 (18.0) → 흡연 가능성을 낮추는 방향으로 작용
 - 혈청크레아티닌 (0.6) → 흡연 가능성을 높이는 방향으로 작용

[권고

c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [ ]:

sample_idx = 0
print(generate_report(sample_idx))

sample_indices = [0, 1, 2, 3, 4]

reports = []
for idx in sample_indices:
    reports.append({
        'id': df.iloc[idx]['id'],
        'report': generate_report(idx)
    })

reports_df = pd.DataFrame(reports)
reports_df.to_csv('../results/sample_reports.csv', index=False, encoding='utf-8-sig')
print("저장 완료: results/sample_reports.csv")

for r in reports:
    print(" ID: {r['id']}")
    print(r['report'])